# Diagnose missing benefits — feedback from load 210 review

Reviewers compared our LLM output against the expected ground truth (`IFP2.dbo.PlanBenefits`) for plan `H3822_002_0` and identified specific benefit IDs that are missing or partial:

- **930** Podiatry — missing entirely
- **1030** Diagnostic/Lab/Radiology — missing entirely
- **990** Outpatient Rehab — missing entirely
- **1200** Kidney Disease — partial (some service types missing)
- **Initial Coverage (710/711)** — missing tiers (generic, specialty)

This notebook isolates ONE plan and identifies for each expected benefit:
1. Does it appear in our LLM output?
2. If not, does the input PBP data contain matchable rows?
3. If input has matchable rows but output doesn't, the LLM is failing on them


## 0. Config — point at your files

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd

# ── Files ─────────────────────────────────────────────────────────────────────
PAYLOAD_PATH = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_payload.json')
OUTPUT_PATH  = Path(r'C:\\Users\\i40212888\\Downloads\\210-20260511T174006Z_output.json')

# ── Plan to focus on ──────────────────────────────────────────────────────────
# From the meeting: contractID=H3822, planID=002 → planid pattern: *_H3822_002_0
# The first match wins. Adjust if you want a different plan.
TARGET_CONTRACT = 'H3822'
TARGET_PLAN     = '002'
TARGET_SEGMENT  = '0'

# Ground-truth benefit IDs from the meeting screenshot (IFP2.PlanBenefits).
# Update this list as you get more accurate data from SQL.
EXPECTED_BENEFIT_IDS = [
    '610', '611', '615', '616', '620',
    '700', '710', '711', '730', '740', '755', '760',
    '800', '810', '820',
    '900', '910', '911', '920', '930',
    '940', '950', '960', '970', '981', '982', '990',
    '1000', '1020', '1030',
    '1050', '1060', '1200',
    '1300', '1301', '1400', '1500',
    '1610', '1700', '1800', '1900',
    '2100',
]

# Specifically called out as missing in the meeting
KNOWN_MISSING = ['930', '990', '1030']
KNOWN_PARTIAL = ['710', '711', '1200']

print(f'Target plan: contract={TARGET_CONTRACT}, plan={TARGET_PLAN}, segment={TARGET_SEGMENT}')
print(f'Expecting {len(EXPECTED_BENEFIT_IDS)} benefit IDs from ground truth')


## 1. Load files and isolate the target plan

In [ ]:
# Load payload (input)
payload_raw = json.loads(PAYLOAD_PATH.read_text(encoding='utf-8'))
payload_rows = payload_raw['pbp'] if isinstance(payload_raw, dict) and 'pbp' in payload_raw else payload_raw

# Load output
output_data = json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
if isinstance(output_data, dict) and 'results' in output_data:
    output_rows = output_data['results']
else:
    output_rows = output_data

print(f'Full payload: {len(payload_rows):,} PBP rows')
print(f'Full output:  {len(output_rows):,} benefit rows\n')

# Isolate target plan in INPUT (matches FileName containing the contract+plan+segment)
target_pattern = f'{TARGET_CONTRACT}-{TARGET_PLAN}'
target_input = [r for r in payload_rows
                if r.get('FileName') and target_pattern in r['FileName']]

# Isolate target plan in OUTPUT (planid matches *_CONTRACT_PLAN_SEGMENT)
target_planid_suffix = f'_{TARGET_CONTRACT}_{TARGET_PLAN}_{TARGET_SEGMENT}'
target_output = [r for r in output_rows
                 if r.get('planid', '').endswith(target_planid_suffix)]

# Diagnostic
input_filenames = sorted({r['FileName'] for r in target_input if r.get('FileName')})
output_planids = sorted({r['planid'] for r in target_output if r.get('planid')})

print(f'Target plan in INPUT:  {len(target_input):,} rows from {len(input_filenames)} FileName(s)')
for fn in input_filenames:
    print(f'   - {fn}')
print(f'\nTarget plan in OUTPUT: {len(target_output)} rows from {len(output_planids)} planid(s)')
for pid in output_planids:
    print(f'   - {pid}')

if not target_input:
    raise SystemExit(f'No input rows matched {target_pattern}. Check TARGET_CONTRACT / TARGET_PLAN.')
if not target_output:
    print(f'\nWARNING: No output rows for this plan. Possibly never produced output, or planid mismatch.')


## 2. What benefit IDs did our LLM produce for this plan?

In [ ]:
out_df = pd.DataFrame(target_output)
if not out_df.empty:
    out_df['benefitid'] = out_df['benefitid'].astype(str)

    bid_counts = out_df.groupby('benefitid').size().to_dict()
    bid_w_servicetype = out_df.groupby('benefitid').agg(
        rows=('benefitid', 'size'),
        distinct_service_types=('serviceTypeID', 'nunique'),
    )
    print(f'Benefit IDs produced for {output_planids[0] if output_planids else "this plan"}:\n')
    display(bid_w_servicetype.sort_index())

    produced_ids = set(bid_counts.keys())
    expected_ids = set(EXPECTED_BENEFIT_IDS)
    missing_from_output = expected_ids - produced_ids
    extra_in_output     = produced_ids - expected_ids

    print(f'\nExpected: {len(expected_ids)} benefits | Produced: {len(produced_ids)} benefits')
    print(f'Missing from output: {len(missing_from_output)}')
    print(f'Extra in output:     {len(extra_in_output)}')
else:
    out_df = pd.DataFrame()
    produced_ids = set()
    missing_from_output = set(EXPECTED_BENEFIT_IDS)
    extra_in_output = set()
    print('No output rows for this plan.')


## 3. For each missing benefit — does the input even contain it?

This is the key diagnostic. For each benefit ID the LLM didn't produce, search the input PBP rows for matchable categories. Distinguishes:
- **INPUT_HAS_NONE** — the data genuinely doesn't have this benefit (not a bug)
- **LLM_MISSED** — the data has matchable rows but the LLM didn't produce output


In [ ]:
# Mirror the filtering logic from build_benefits.py v4
BENEFIT_TARGETS = {
    '610':  ([],             ['Health Plan Deductible', 'Medical Deductible', 'Annual Plan Deductible']),
    '611':  ([],             ['Drug Deductible', 'Rx Deductible', 'Enter Deductible Amount']),
    '615':  ([],             ['Drug Monthly Premium', 'Rx Premium', 'Part D Premium']),
    '616':  ([],             ['Part B Premium', 'Part B Reduction', 'Part B giveback']),
    '620':  ([],             ['MOOP', 'Max Enrollee Cost', 'Out of Pocket', 'OOP']),
    '700':  ([],             ['Rx Tier', 'Formulary Tier', 'Tier Names']),
    '710':  ([],             ['Initial Coverage Phase', 'Rx Setup']),
    '711':  ([],             ['Retail Pharmacy', 'Initial Coverage Phase']),
    '730':  ([],             ['Catastrophic Coverage']),
    '740':  ([],             ['Formulary Exception']),
    '755':  ([],             ['Preferred Mail Order']),
    '760':  ([],             ['Standard Mail Order']),
    '800':  (['1a'],         ['Inpatient Hospital']),
    '810':  (['1b'],         ['Inpatient Mental Health', 'Inpatient Psychiatric']),
    '820':  (['2'],          ['Skilled Nursing', 'SNF']),
    '900':  (['7a'],         ['Primary Care', 'PCP']),
    '910':  (['7b', '7d'],   ['Specialist', 'Specialty Care']),
    '911':  (['7d', '7j'],   ['Telehealth', 'Telemedicine', 'Virtual', 'Additional Telehealth']),
    '920':  (['8'],          ['Chiropractic']),
    '930':  (['9a'],         ['Podiatry']),
    '940':  (['4a'],         ['Outpatient Mental Health']),
    '950':  (['4b'],         ['Substance Abuse']),
    '960':  (['4c','9a1','9a2','9b','9d'], ['Outpatient Surgery', 'Outpatient Hospital', 'Ambulatory Surgical']),
    '970':  (['5','5a','5b'],['Ambulance']),
    '981':  (['4a','6a'],    ['Emergency Services']),
    '982':  (['4b','6b'],    ['Urgent Care', 'Urgently Needed']),
    '990':  (['10','5b'],    ['Outpatient Rehabilitation', 'Physical Therapy']),
    '1000': (['11a'],        ['Durable Medical Equipment', 'DME']),
    '1020': (['11c'],        ['Diabetic', 'Diabetes']),
    '1030': (['3','8a2','8b2'], ['Diagnostic', 'X-Ray', 'Lab Services', 'Radiology']),
    '1050': (['13b','14c4'], ['Fitness']),
    '1060': (['13c'],        ['Meals']),
    '1200': (['12'],         ['Kidney', 'Renal', 'Dialysis']),
    '1300': (['16a'],        ['Preventive Dental', 'Medicare Dental']),
    '1301': (['16b','16c1','16c'], ['Comprehensive Dental', 'Restorative']),
    '1400': (['18a','18b','18b1'], ['Hearing Exam', 'Hearing Aid', 'Fitting/Evaluation']),
    '1500': (['17a','17a1','17b','17b2','17b3'], ['Eye Exam', 'Eyewear', 'Eyeglasses', 'Contact Lenses']),
    '1610': (['11b'],        ['Prosthetic']),
    '1700': (['14a','14b','14c'], ['Preventive', 'Wellness Visit']),
    '1800': (['15'],         ['Transportation']),
    '1900': (['13a'],        ['Acupuncture']),
    '2100': (['13b','13e'],  ['Over-the-Counter', 'OTC']),
}

def row_matches_target(row, codes, keywords):
    cat = (row.get('category') or '').lower()
    for c in codes:
        if f'({c})' in cat or f'({c}1)' in cat or f'({c}2)' in cat:
            return True
    for kw in keywords:
        if kw.lower() in cat:
            return True
    return False

# Analyze each missing benefit
diagnosis_rows = []
for bid in sorted(missing_from_output, key=lambda x: int(x) if x.isdigit() else 99999):
    codes, keywords = BENEFIT_TARGETS.get(bid, ([], []))
    matched_input = [r for r in target_input if row_matches_target(r, codes, keywords)]
    distinct_cats = sorted({r.get('category', '') for r in matched_input})

    if not matched_input:
        verdict = 'INPUT_HAS_NONE'
    else:
        verdict = 'LLM_MISSED'

    diagnosis_rows.append({
        'benefitid':         bid,
        'matched_input_rows': len(matched_input),
        'distinct_categories': len(distinct_cats),
        'verdict':           verdict,
        'sample_category':   distinct_cats[0][:80] if distinct_cats else '',
    })

diag_df = pd.DataFrame(diagnosis_rows)
print(f'Missing benefits diagnosis for plan {target_planid_suffix.lstrip("_")}:\n')
display(diag_df)

n_input_none = (diag_df['verdict'] == 'INPUT_HAS_NONE').sum() if len(diag_df) else 0
n_llm_missed = (diag_df['verdict'] == 'LLM_MISSED').sum() if len(diag_df) else 0

print(f'\nINPUT_HAS_NONE: {n_input_none}  (data genuinely missing — not a code bug)')
print(f'LLM_MISSED:     {n_llm_missed}  (input has rows, LLM didn\'t produce output)')


## 4. Deep dive on the specifically-called-out missing benefits

Show actual input rows for 930 (Podiatry), 990 (Outpatient Rehab), 1030 (Diagnostic/Lab/Radiology). If these have lots of input rows but no output, that's a clear LLM/filter problem.

In [ ]:
for bid in KNOWN_MISSING:
    codes, keywords = BENEFIT_TARGETS.get(bid, ([], []))
    matched = [r for r in target_input if row_matches_target(r, codes, keywords)]
    print(f'─── benefit {bid} ────────────────────────────────────────────')
    print(f'Match codes:    {codes}')
    print(f'Match keywords: {keywords}')
    print(f'Matched input:  {len(matched)} rows')
    if matched:
        cats = Counter(r.get('category', '') for r in matched)
        print(f'Top input categories:')
        for cat, n in cats.most_common(5):
            print(f'  ({n:>3})  {cat[:90]}')

        # Show a representative sample of the actual data
        print(f'\nSample input row:')
        sample = matched[0]
        for k, v in sample.items():
            print(f'  {k}: {v}')
    else:
        print(f'⚠ No input rows match — either filter is wrong or this benefit truly absent')
    print()


## 5. Deep dive on partial benefits (710/711 missing tiers, 1200 missing service types)

For benefits the LLM produced rows for but where reviewers flagged missing service types, show the service types we produced vs what's expected.

In [ ]:
# Expected service types (from meeting context)
EXPECTED_TIERS_FOR_711 = ['Preferred Generic', 'Generic', 'Preferred Brand', 'Non-Preferred Drug', 'Specialty Tier']
EXPECTED_TIERS_FOR_710 = ['Preferred Generic', 'Generic', 'Preferred Brand', 'Non-Preferred Drug', 'Specialty Tier']

for bid in KNOWN_PARTIAL:
    rows_for_bid = [r for r in target_output if str(r.get('benefitid')) == bid]
    print(f'─── benefit {bid} ────────────────────────────────────────────')
    print(f'Rows produced: {len(rows_for_bid)}')
    if rows_for_bid:
        service_types = sorted({(r.get('serviceTypeID'), r.get('serviceTypeDesc'))
                                for r in rows_for_bid}, key=lambda x: (str(x[0])))
        print(f'Service types produced:')
        for stid, stdesc in service_types:
            print(f'  {stid}: {stdesc}')

        # What's missing for tier-based benefits?
        if bid in ('710', '711'):
            produced_descs = {sd for _, sd in service_types if sd}
            missing_tiers = [t for t in EXPECTED_TIERS_FOR_711
                             if not any(t.lower() in pd.lower() for pd in produced_descs)]
            if missing_tiers:
                print(f'\n⚠ Expected tiers missing from output: {missing_tiers}')
            else:
                print(f'\n✓ All expected tiers present')
    print()


## 6. Bottom-line summary — what to report back

Use this output verbatim when responding to the reviewers.

In [ ]:
print('=' * 75)
print(f'Investigation: plan H3822_002_0 — load 210')
print('=' * 75)
print()
print(f'Expected benefits (from IFP2 ground truth): {len(EXPECTED_BENEFIT_IDS)}')
print(f'LLM-produced benefits for this plan:        {len(produced_ids)}')
print(f'Missing from LLM output:                    {len(missing_from_output)}')
print()
print('BREAKDOWN OF MISSING BENEFITS:')
print(f'  INPUT_HAS_NONE: {n_input_none:>3}  — data not in PBP source; expected from plan registry or')
print(f'                       supplemental files. Not an LLM bug.')
print(f'  LLM_MISSED:     {n_llm_missed:>3}  — input HAS matchable rows; LLM did not produce output.')
print(f'                       Need to fix BENEFIT_TARGETS or per-benefit prompt.')
print()
print('SPECIFICALLY CALLED OUT:')
for bid in KNOWN_MISSING + KNOWN_PARTIAL:
    diag_match = diag_df[diag_df['benefitid'] == bid]
    if len(diag_match):
        d = diag_match.iloc[0]
        print(f'  {bid:>5}  {d["matched_input_rows"]:>4} input rows  →  {d["verdict"]}')
    else:
        n_rows = sum(1 for r in target_output if str(r.get('benefitid')) == bid)
        if n_rows > 0:
            print(f'  {bid:>5}  {n_rows} rows produced (partial — see section 5)')
        else:
            print(f'  {bid:>5}  not in output, not in expected list either')

# Save findings for sharing
findings_path = OUTPUT_PATH.parent / f'missing_benefits_diagnosis_H3822_002_0.csv'
if len(diag_df):
    diag_df.to_csv(findings_path, index=False)
    print(f'\nSaved per-benefit diagnosis to: {findings_path}')


## 7. Optional — query the SQL ground truth directly

If you have read access to the IFP2 database, you can replace the hardcoded `EXPECTED_BENEFIT_IDS` list with a live query. This template uses pyodbc.

In [ ]:
# Uncomment and configure if you want to query SQL directly
# import pyodbc
#
# CONN_STR = (
#     'DRIVER={ODBC Driver 17 for SQL Server};'
#     'SERVER=your-server;'
#     'DATABASE=IFP2;'
#     'Trusted_Connection=yes;'
# )
#
# query = '''
#     SELECT DISTINCT bv.benefitName, pb.*
#     FROM IFP2.dbo.PlanBenefits pb WITH (NOLOCK)
#     JOIN IFP2.dbo.benefitsView bv WITH (NOLOCK)
#       ON bv.periodID = 336 AND bv.benefitID = pb.benefitID
#     WHERE pb.endingPeriodID = 336
#       AND pb.planID = ?
# '''
#
# target_planid_for_sql = f'NMM_H{TARGET_CONTRACT[1:]}_{TARGET_PLAN}_{TARGET_SEGMENT}'
# # Adjust the planid prefix to whatever the IFP2 schema actually uses
#
# with pyodbc.connect(CONN_STR) as conn:
#     gt = pd.read_sql(query, conn, params=[target_planid_for_sql])
#
# print(f'Ground truth: {len(gt)} rows for {target_planid_for_sql}')
# display(gt[['benefitID', 'benefitName']].drop_duplicates())
print('Uncomment the cell above to query SQL directly')
